In [9]:
import pandas as pd 
import numpy as np

df = pd.read_csv('ncaa_data.csv')

In [10]:
df.head()

,game_id,period,game_clock,home_name,home_score,away_score,away_name,home_score_differential,total_seconds_remaining,home_team_wins
0,000872e5-f02a-4b64-ac73-b6b1a7ad10ec,1,19:39,Gators,19.0,9.0,Bulldogs,10.0,2379,0
1,000872e5-f02a-4b64-ac73-b6b1a7ad10ec,1,19:37,Gators,19.0,9.0,Bulldogs,10.0,2377,0
2,000872e5-f02a-4b64-ac73-b6b1a7ad10ec,1,19:22,Gators,19.0,10.0,Bulldogs,9.0,2362,0
3,000872e5-f02a-4b64-ac73-b6b1a7ad10ec,1,19:22,Gators,19.0,10.0,Bulldogs,9.0,2362,0
4,000872e5-f02a-4b64-ac73-b6b1a7ad10ec,1,19:22,Gators,19.0,10.0,Bulldogs,9.0,2362,0


In [11]:
df.columns

Index(['game_id', 'period', 'game_clock', 'home_name', 'home_score',
       'away_score', 'away_name', 'home_score_differential',
       'total_seconds_remaining', 'home_team_wins'],
      dtype='object')

In [12]:
# Defining the feature (X) 
X = df.drop(columns=['home_team_wins'])
# Defining target variable 
y = df['home_team_wins']

In [13]:

from sklearn.model_selection import GroupShuffleSplit
# Initializing my split 80/20
gss = GroupShuffleSplit(n_splits=1,train_size = 0.8,test_size= 0.2, random_state=42)
## Here we are defining the random train/test split. Do 1 split of 80/20. Random State set to 42 to split the same way in the future 

# Getting the indeces of the split, grouping by game_id 
train_idx, test_idx = next(gss.split(X, y , groups=X['game_id']))
## gss.split is splitting the inputted data on the basis of game_id
## next() is telling to do the spit. The result is the row numbers for the training set and the testing set 
## The package contains two lists so we provide two varaible names seperated by a comma. The first list of numbers goes to train_idx, the second to test_idx 


In [14]:
## Creating the final leakage free training and testing sets 
## train_idx and test_idx are just arrays of row numbers, do not contian actual data. 
## iloc stands for indeger location. It looks up rows by their numberic order starting at 0 
## This code is telling pandas to go into X DataFrame, look at the integer location of every row, and extract only 
### the rows whose numerical pisition match ht enumbers in my train_idx. 
X_train = X.iloc[train_idx] 
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [15]:
## Dropping the remaining non-numeric and identifier columns before training 
cols_to_drop = ['game_id', 'home_name', 'away_name', 'game_clock']
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)


In [16]:
X_train

,period,home_score,away_score,home_score_differential,total_seconds_remaining
0,1,19.0,9.0,10.0,2379
1,1,19.0,9.0,10.0,2377
2,1,19.0,10.0,9.0,2362
3,1,19.0,10.0,9.0,2362
4,1,19.0,10.0,9.0,2362
...,...,...,...,...,...
86240,1,9.0,14.0,-5.0,1394
86241,1,9.0,14.0,-5.0,1394
86242,1,9.0,14.0,-5.0,1375
86243,1,9.0,14.0,-5.0,1373


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

baseline_model = LogisticRegression(max_iter=1000, random_state=42)

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test) ## these are hard predictions 1 or 0s

baseline_accuracy = accuracy_score(y_test, y_pred)
print(f"Baseline Model Accuracy: {baseline_accuracy * 100:.2f}%")

Baseline Model Accuracy: 80.40%


In [18]:
from sklearn.metrics import roc_auc_score

## Need to get probabilities to have an area under the curve
## predict proba returns two columns, [probability of loss, probability of win]
## use [:, 1] to only grab the second column 

y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]

# Calculate the AUC score 
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"Baseline Model AUC: {auc_score:.4f}")


Baseline Model AUC: 0.9003


In [20]:
import xgboost as xgb 
from sklearn.metrics import roc_auc_score

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    learning_rate =0.1, 
    max_depth =5, 
    n_estimator=100
)

xgb_model.fit(X_train, y_train)

xgb_y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_auc_score = roc_auc_score(y_test, xgb_y_pred_proba)
print(f"XGB AUC Score: {xgb_auc_score:.4f}")

XGB AUC Score: 0.9038


/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [15:45:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [21]:
xgb_train_pred_proba = xgb_model.predict_proba(X_train)[:, 1]

train_auc_score = roc_auc_score(y_train, xgb_train_pred_proba)

print(f"Train XGB AUC Score: {train_auc_score:.4f}")
print(f"XGB AUC Score: {xgb_auc_score:.4f}")

Train XGB AUC Score: 0.8883
XGB AUC Score: 0.9038
